# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/star-instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

In [4]:
from llm.text_generator import TextGenerator

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/message_generator.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Constants

In [5]:
TAWJEEH_DATASET_NAME = 'opus-100'
HF_EXPERIMENTAL_DATASET_NAME = 'KFUPM-JRCAI/opus-100_ar_en_experimental'
TASK_NAME='machine_translation'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"

In [6]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [7]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [8]:
len(prompts)

365

In [9]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

## Finetuning

### Get the dataset prompts

In [10]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

10

In [11]:
SELECTED_PROMPTS_IDS = [
    14684,
    14688,
    14680,
    14682,
    14640,
]

In [12]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [13]:
import datasets

In [14]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['ar', 'en'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['ar', 'en'],
        num_rows: 2000
    })
})

In [15]:
hf_exp_dataset = hf_exp_dataset.map(lambda example: {'translation':{'en':example['en'], 'ar':example['ar']}}, remove_columns=['en', 'ar'])
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

### Merge the prompts

In [16]:
from jinja2 import Environment, StrictUndefined

In [17]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [18]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

### Perform prompt-merge on one example prompt, for experimentation

In [19]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

Task: Carefully translate the following English sentence into Arabic, ensuring that the translation reflects the original intent and context.

English Sentence: Okay.

Guidelines:
1. Accurate Meaning: Ensure the translation conveys the exact meaning of the English sentence without losing any details.
2. Cultural and Contextual Fit: Consider any cultural or contextual nuances to translate feel natural in Arabic.
3. Tone Consistency: Maintain the same tone and style as the original, whether formal, casual, or neutral.
حسناً؟


In [20]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

6000.0

In [21]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/30000 [00:00<?, ?it/s]

rending Task: Carefully translate the following English sentence into Arabic, ensuring that the translation reflects the original intent and context.

English Sentence: {{ translation['en'] }}

Guidelines:
1. Accurate Meaning: Ensure the translation conveys the exact meaning of the English sentence without losing any details.
2. Cultural and Contextual Fit: Consider any cultural or contextual nuances to translate feel natural in Arabic.
3. Tone Consistency: Maintain the same tone and style as the original, whether formal, casual, or neutral.

|||
{{ translation['ar'] }} sample index: 0


rending Task: Translate the following English sentence into Arabic by reasoning step by step.

English Sentence: {{ translation['en'] }}

Step 1: Break Down the Meaning
Start by carefully analyzing the sentence. Identify the subject, verb, and key details to fully understand its meaning in context.

Step 2: Identify Important Words
Highlight the keywords or phrases that carry significant meaning, and think about their most accurate Arabic equivalents.

Step 3: Construct the Translation
Using the keywords and your understanding of the sentence’s context, construct a smooth, grammatically correct translation in Arabic. Ensure it sounds natural and conveys the full meaning of the English sentence.

|||
{{ translation['ar'] }} sample index: 6000


rending Task: Translate the following sentence from English to Arabic as accurately as possible.
English Sentence: {{ translation['en'] }}

Response:
Provide the translation in Arabic.

|||
{{ translation['ar'] }} sample index: 12000


rending Task: Translate the following sentence from English to Arabic.

English Sentence: {{ translation['en'] }}

Translation:
Provide the correct translation in Arabic.
|||
{{ translation['ar'] }} sample index: 18000


rending Translate the text from English to Arabic: {{translation ['en']}}
|||
{{translation['ar']}} sample index: 24000


30000

## Finetune the LLM

In [22]:
GLOBAL_SEED = 42

In [23]:
import random
random.seed(GLOBAL_SEED)

In [24]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Qwen3Initializer, LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [25]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Qwen3Initializer(),
)
llm_loader

In [26]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/config.json


`torch_dtype` is deprecated! Use `dtype` instead!


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

loading weights file /raid_storage/shared_models/Qwen3-8B-Base/model.safetensors.index.json


Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.


...

loading file tokenizer_config.json


loading file chat_template.jinja


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/generation_config.json


Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



In [27]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    prefix = '\n'.join(sample_lines[:-1])
    prefix += '\nTranslation:'
    prefix = prefix.strip()
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(27000,
 3000,
 [('Task: Carefully translate the following English sentence into Arabic, ensuring that the translation reflects the original intent and context.\n\nEnglish Sentence: Why do you ask?\n\nGuidelines:\n1. Accurate Meaning: Ensure the translation conveys the exact meaning of the English sentence without losing any details.\n2. Cultural and Contextual Fit: Consider any cultural or contextual nuances to translate feel natural in Arabic.\n3. Tone Consistency: Maintain the same tone and style as the original, whether formal, casual, or neutral.\nTranslation:',
   ' لمَ تسألان؟ لقد مات'),
  ("Task: Translate the following sentence from English to Arabic as accurately as possible.\nEnglish Sentence: If it's any consolation, the, uh... the guy stole my Wall Street Journal once.\n\nResponse:\nProvide the translation in Arabic.\nTranslation:",
   ' إن كان في هذا أي عزاء ٍ لكِ فإن ذلك الرجل قد سرق مني مجلة وول ستريت " في إحدى المرات "'),
  ('Task: Translate the following English sente

In [28]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=1,
    eval_batch_size=4,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}',
    early_stopping_patience=10,
    eval_steps=500,
)

PyTorch: setting up devices


The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/llm_trainer.py:83: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.


Using auto half precision backend



***** Running Evaluation *****


...

  Num examples = 3000


  Batch size = 4


{'eval_loss': 1.6755893230438232, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 71.924, 'eval_samples_per_second': 41.711, 'eval_steps_per_second': 10.428, 'epoch': 0.4074074074074074}




Training completed. Do not forget to share your model on huggingface.co/models =)




1.6742252111434937

In [29]:
exit()